## Trabalho Final: Construção de um Pipeline em PySpark

Este trabalho visa à reprodução de um pipeline de dados simplificado, utilizando como fonte uma base de dados proveniente do portal Dados Públicos (https://dados.gov.br/).

---

A base de dados designada para cada estudante foi previamente selecionada e está detalhada na seguinte [planilha](https://docs.google.com/spreadsheets/d/1QZmmK4Xsp8woSdCZrQT78y1p8w9I-rSB/edit?usp=sharing&ouid=105817500216613802640&rtpof=true&sd=true). Nesta planilha, o estudante encontrará seu `Nome`, o `Nome da base` com a qual deverá trabalhar e o `link` para download e identificação do conjunto de dados.


## Orientações para o Desenvolvimento do Projeto

Desenvolva um script em Python, utilizando a biblioteca PySpark, que atenda aos seguintes requisitos:

1.  Realizar a leitura de um arquivo no formato CSV ou TXT ou Parquet.
2.  Converter todos os valores vazios (empty values) para o tipo Nulo (NULL).
3.  Aplicar a tipagem de dados correta para *todas* as colunas do DataFrame.
4.  Definir e aplicar um valor *default* para *todas* as colunas do DataFrame, onde aplicável.
5.  Salvar o DataFrame resultante no formato Parquet, utilizando uma coluna específica para particionamento.
6.  O script desenvolvido deve ser executável por meio do comando `spark-submit`.

---

## Considerações Relevantes

1.  Priorize a aplicação dos conceitos e técnicas abordados em sala de aula.
2.  A avaliação do trabalho será fundamentada nos seis critérios estabelecidos na seção anterior, verificando a capacidade do código em implementar cada um dos tópicos.
3.  Caso opte por desenvolver o projeto em um notebook Python, é imprescindível a utilização do comando mágico `%%writefile` para a criação do arquivo `.py` final.

---

## Critérios de Avaliação e Políticas Acadêmicas

1.  Projetos que demonstrarem o uso de ferramentas de Inteligência Artificial para a geração de código terão sua nota zerada.
2.  Apresentação de trabalhos idênticos resultará na anulação da nota para todos os envolvidos.
3.  Trabalhos desenvolvidos em linguagens de programação diferentes de Python (e.g., Scala, R ou Java) não serão aceitos e terão sua nota zerada.


### Revisão

In [4]:
%pip install pyspark duckdb


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import SparkContext, SparkConf
import time

Config = SparkConf()
Config.set("spark.sql.repl.eagerEval.enabled", True)
Config.set("spark.sql.repl.eagerEval.maxNumRows", "20")
Config.set("spark.sql.repl.eagerEval.truncate", "-1")
Config.set("spark.driver.memory","10G")
Config.set("spark.memory.fraction", 0.9)
Config.set("spark.sql.adaptive.enabled", "true")
Config.set("spark.sql.adaptive.join.enabled", "true")
Config.set('spark.sql.legacy.allowNonEmptyLocationInCTAS','true')
Config.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
Config.set("spark.sql.sources.partitionOverwriteMode","dynamic")
Config.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

# Config.set("spark.sql.shuffle.partitions", 100)
# Config.set("spark.default.parallelism", 200)

spark = SparkSession.builder.config(conf=Config).master("local[*]").appName("TrabalhoFinal").getOrCreate()

In [6]:
from IPython.core.magic import (register_line_magic, register_cell_magic,register_line_cell_magic)

@register_line_cell_magic('sql')
def sparksql(line, cell=None):
    "Esse Magic funciona com  %sql e %%sql"
    try: spark
    except NameError: print('Spark instance is not defined')
    else:
        spark.conf.set('spark.sql.repl.eagerEval.enabled','true')
        spark.conf.set('spark.sql.repl.eagerEval.maxNumRows', 100)
        spark.conf.set('spark.sql.repl.eagerEval.truncate',-1)
        if cell is None:
            result = spark.sql(line)
            return result
        else:
            result = spark.sql(cell)
            return result

In [7]:
path = "/Users/fabiokishino/Documents/Dev/pos-data-science/big-data/Trabalho/discentes-egressos-2025.csv"

df_csv = (
    spark.read
    .option("sep", ";")
    .option("quote", '"')
    .option("header", "true")
    .option("encoding", "ISO-8859-1")
    .csv(path)
)

In [8]:
df_csv.write.parquet("parquet/", mode='overwrite')

In [9]:
df = spark.read.parquet("parquet/")

In [10]:
df.printSchema()

root
 |-- matricula: string (nullable = true)
 |-- nome_discente: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- ano_conclusao: string (nullable = true)
 |-- periodo_conclusao: string (nullable = true)
 |-- ano_ingresso: string (nullable = true)
 |-- periodo_ingresso: string (nullable = true)
 |-- id_curso: string (nullable = true)
 |-- nome_curso: string (nullable = true)
 |-- modalidade_educacao: string (nullable = true)
 |-- forma_ingresso: string (nullable = true)
 |-- tipo_discente: string (nullable = true)
 |-- nivel_ensino: string (nullable = true)
 |-- id_unidade: string (nullable = true)
 |-- nome_unidade: string (nullable = true)
 |-- id_unidade_gestora: string (nullable = true)
 |-- nome_unidade_gestora: string (nullable = true)



In [11]:
df = df.select(*[nullif(df[column_name], lit('')).alias(column_name) for column_name in df.columns])


In [ ]:
df

In [12]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

In [14]:
df.select("ano_conclusao").distinct().orderBy("ano_ingresso").show()

+-------------+
|ano_conclusao|
+-------------+
|         2025|
+-------------+



In [ ]:
##  Colunas
# matricula
# nome_discente
# sexo
# ano_conclusao
# periodo_conclusao
# ano_ingresso
# periodo_ingresso
# id_curso
# nome_curso
# modalidade_educacao
# forma_ingresso
# tipo_discente
# nivel_ensino
# id_unidade
# nome_unidade
# id_unidade_gestora
# nome_unidade_gestora


df = df.select(
    df["matricula"].try_cast(IntegerType()).alias("matricula"),
    df["nome_discente"].try_cast(StringType()).alias("nome_discente"),
    df["sexo"].try_cast(StringType()).alias("sexo"),
    df["ano_conclusao"].try_cast(IntegerType()).alias("ano_conclusao"),
    df["periodo_conclusao"].try_cast(IntegerType()).alias("periodo_conclusao"),
    df["ano_ingresso"].try_cast(IntegerType()).alias("ano_ingresso"),
    df["periodo_ingresso"].try_cast(IntegerType()).alias("periodo_ingresso"),
    df["id_curso"].try_cast(IntegerType()).alias("id_curso"),
    df["nome_curso"].try_cast(StringType()).alias("nome_curso"),
    df["modalidade_educacao"].try_cast(StringType()).alias("modalidade_educacao"),
    df["forma_ingresso"].try_cast(StringType()).alias("forma_ingresso"),
    df["tipo_discente"].try_cast(StringType()).alias("tipo_discente"),  
    df["nivel_ensino"].try_cast(StringType()).alias("nivel_ensino"),
    df["id_unidade"].try_cast(IntegerType()).alias("id_unidade"),
    df["nome_unidade"].try_cast(StringType()).alias("nome_unidade"),
    df["id_unidade_gestora"].try_cast(IntegerType()).alias("id_unidade_gestora"),
    df["nome_unidade_gestora"].try_cast(StringType()).alias("nome_unidade_gestora"),
    current_timestamp().alias("update_date")
)
df.createOrReplaceTempView("raw_data")

df = df.na.fill({
  'matricula': -1,
  'nome_discente': "N/A",
  'sexo': "N/A",
  'ano_conclusao': -1,
  'periodo_conclusao': -1,
  'ano_ingresso': -1,
  'periodo_ingresso': -1,
  'id_curso': -1,
  'nome_curso': "N/A",
  'modalidade_educacao': "N/A",
  'forma_ingresso': "N/A",
  'tipo_discente': "N/A",
  'nivel_ensino': "N/A",
  'id_unidade': -1,
  'nome_unidade': "N/A",
  'id_unidade_gestora': -1,
  'nome_unidade_gestora': "N/A",
})


In [15]:
df.repartition("ano_ingresso").write.partitionBy("ano_ingresso").mode("overwrite").parquet("partitioned_data/")

In [ ]:
%%writefile trabalho_final_fabio_kishino.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import SparkConf

Config = SparkConf()
Config.set("spark.sql.repl.eagerEval.enabled", True)
Config.set("spark.sql.repl.eagerEval.maxNumRows", "20")
Config.set("spark.sql.repl.eagerEval.truncate", "-1")
Config.set("spark.driver.memory","10G")
Config.set("spark.memory.fraction", 0.9)
Config.set("spark.sql.adaptive.enabled", "true")
Config.set("spark.sql.adaptive.join.enabled", "true")
Config.set('spark.sql.legacy.allowNonEmptyLocationInCTAS','true')
Config.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
Config.set("spark.sql.sources.partitionOverwriteMode","dynamic")
Config.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

spark = SparkSession.builder.config(conf=Config).master("local[*]").appName("TrabalhoFinal").getOrCreate()

path = "/Users/fabiokishino/Documents/Dev/pos-data-science/big-data/Trabalho/discentes-egressos-2025.csv"

df_csv = (
    spark.read
    .option("sep", ";")
    .option("quote", '"')
    .option("header", "true")
    .option("encoding", "ISO-8859-1")
    .csv(path)
)

df_csv.write.parquet("parquet/", mode='overwrite')

df = spark.read.parquet("parquet/")

df = df.select(*[nullif(df[column_name], lit('')).alias(column_name) for column_name in df.columns])

df = df.select(
    df["matricula"].try_cast(LongType()).alias("matricula"),
    df["nome_discente"].try_cast(StringType()).alias("nome_discente"),
    df["sexo"].try_cast(StringType()).alias("sexo"),
    df["ano_conclusao"].try_cast(IntegerType()).alias("ano_conclusao"),
    df["periodo_conclusao"].try_cast(IntegerType()).alias("periodo_conclusao"),
    df["ano_ingresso"].try_cast(IntegerType()).alias("ano_ingresso"),
    df["periodo_ingresso"].try_cast(IntegerType()).alias("periodo_ingresso"),
    df["id_curso"].try_cast(IntegerType()).alias("id_curso"),
    df["nome_curso"].try_cast(StringType()).alias("nome_curso"),
    df["modalidade_educacao"].try_cast(StringType()).alias("modalidade_educacao"),
    df["forma_ingresso"].try_cast(StringType()).alias("forma_ingresso"),
    df["tipo_discente"].try_cast(StringType()).alias("tipo_discente"),  
    df["nivel_ensino"].try_cast(StringType()).alias("nivel_ensino"),
    df["id_unidade"].try_cast(IntegerType()).alias("id_unidade"),
    df["nome_unidade"].try_cast(StringType()).alias("nome_unidade"),
    df["id_unidade_gestora"].try_cast(IntegerType()).alias("id_unidade_gestora"),
    df["nome_unidade_gestora"].try_cast(StringType()).alias("nome_unidade_gestora"),
    current_timestamp().alias("update_date")
)
df.createOrReplaceTempView("raw_data")

df = df.na.fill({
  'matricula': -1,
  'nome_discente': "N/A",
  'sexo': "N/A",
  'ano_conclusao': -1,
  'periodo_conclusao': -1,
  'ano_ingresso': -1,
  'periodo_ingresso': -1,
  'id_curso': -1,
  'nome_curso': "N/A",
  'modalidade_educacao': "N/A",
  'forma_ingresso': "N/A",
  'tipo_discente': "N/A",
  'nivel_ensino': "N/A",
  'id_unidade': -1,
  'nome_unidade': "N/A",
  'id_unidade_gestora': -1,
  'nome_unidade_gestora': "N/A",
})

df.repartition("ano_ingresso").write.partitionBy("ano_ingresso").mode("overwrite").parquet("partitioned_data/")

print("Sucesso!")

Overwriting trabalho_final_fabio_kishino.py
